# Neural Voice Identity Control & Deepfake Audio Analysis
**Disertație 2026 — Demo interactiv**

---

**⚠️ DISCLAIMER:**
Acest sistem este construit exclusiv în scop academic, pentru demonstrarea capacităților și limitelor
tehnologiilor de sinteză vocală și detecție deepfake audio. Vocile simulate sunt ale unor persoane publice
ale căror înregistrări sunt disponibile public. Sistemul **nu este** destinat înșelăciunii, uzului comercial
sau distribuției publice.

---

**Cum rulezi:**
1. Asigură-te că runtime-ul este T4 GPU (Runtime → Change runtime type)
2. Rulează toate celulele: Runtime → Run all
3. Accesează URL-ul Gradio generat la final

**Prerequisite:** Modelele fine-tuned (`.pth`) trebuie să existe în Google Drive.
Dacă nu ai antrenat încă, rulează `02_rvc_finetune.ipynb` prima dată.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import torch
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (no GPU)"}')

In [ ]:
# Instalare dependinte
import subprocess, sys

def pip_install(pkg, extra_args=[]):
    r = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', pkg] + extra_args,
        capture_output=True, text=True
    )
    status = 'OK' if r.returncode == 0 else 'FAIL'
    print(f'  [{status}] {pkg}')
    return r.returncode == 0

print('Instalare pachete...')

for pkg in [
    'gradio>=4.7.0',
    'librosa==0.9.2',
    'soundfile',
    'pydub',
    'transformers',
    'accelerate',
    'matplotlib',
    'plotly',
    'ffmpeg-python',
    'av',
    'scipy',
    'numba',
    'einops',
    'praat-parselmouth>=0.4.2',
    'torchcrepe',
    'faiss-cpu',
    'local_attention',
]:
    pip_install(pkg)

if not pip_install('pyworld'):
    pip_install('pyworld', ['--no-build-isolation'])

subprocess.run(['apt-get', 'install', '-q', '-y', 'ffmpeg'], capture_output=True)
print('Dependinte instalate.')

In [ ]:
# Setup RVC (clone if not present)
import os
import sys

RVC_DIR = '/content/RVC'

if not os.path.exists(RVC_DIR):
    print('Cloning RVC repository...')
    !git clone https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI.git {RVC_DIR} -q
    print('RVC cloned.')
else:
    print('RVC already present.')

sys.path.insert(0, RVC_DIR)

# Download HuBERT + RMVPE if needed
assets = [
    ('https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/hubert_base.pt',
     f'{RVC_DIR}/assets/hubert/hubert_base.pt'),
    ('https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/rmvpe.pt',
     f'{RVC_DIR}/assets/rmvpe/rmvpe.pt'),
]
for url, dest in assets:
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    if not os.path.exists(dest):
        print(f'Downloading {os.path.basename(dest)}...')
        !wget -q '{url}' -O '{dest}'
    else:
        print(f'  OK: {os.path.basename(dest)}')
print('RVC setup complete.')

In [ ]:
# Mock fairseq cu transformers HuBERT
# RVC importa fairseq pentru a incarca HuBERT la inferenta.
# fairseq nu merge pe Python 3.12, deci cream un modul fals
# cu aceeasi interfata API dar care foloseste transformers intern.
import sys, types, torch

def _install_fairseq_mock():
    from transformers import HubertModel, Wav2Vec2FeatureExtractor

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    is_half = torch.cuda.is_available()

    print('Incarcare HuBERT pentru inferenta (transformers)...')
    _extractor = Wav2Vec2FeatureExtractor.from_pretrained('facebook/hubert-base-ls960')
    _hubert = HubertModel.from_pretrained('facebook/hubert-base-ls960').to(device)
    if is_half:
        _hubert = _hubert.half()
    _hubert.eval()
    print('HuBERT incarcat.')

    class FakeHubertModel:
        """Interfata fairseq, backend transformers. Layer 9 = acelasi ca la antrenare."""
        def extract_features(self, source, padding_mask=None, output_layer=9):
            with torch.no_grad():
                src = source.half() if is_half else source.float()
                out = _hubert(src, output_hidden_states=True)
                feats = out.hidden_states[9].float()
            return feats, None

        def final_proj(self, x): return x
        def to(self, d): return self
        def half(self): return self
        def float(self): return self
        def eval(self): return self

    _fake_model = FakeHubertModel()

    def _load_fake(*args, **kwargs):
        return [_fake_model], None, None

    # Inregistreaza modulele fake in sys.modules
    for mod_name in ['fairseq', 'fairseq.checkpoint_utils',
                     'fairseq.data', 'fairseq.models',
                     'fairseq.tasks', 'fairseq.criterions']:
        sys.modules[mod_name] = types.ModuleType(mod_name)

    sys.modules['fairseq.checkpoint_utils'].load_model_ensemble_and_task = _load_fake
    sys.modules['fairseq'].checkpoint_utils = sys.modules['fairseq.checkpoint_utils']
    print('Mock fairseq OK.')

_install_fairseq_mock()

## Configurare voci

Verifica ca modelele `.pth` exista pe Drive inainte sa continui.

In [ ]:
from pathlib import Path

# ============================================================
# CONFIGURARE
# ============================================================
DRIVE_BASE = '/content/drive/MyDrive/disertatie'
MODELS_DIR = Path(f'{DRIVE_BASE}/models')

VOICE_CONFIG = {
    'voice_1': {
        'name': 'Călin Georgescu',
        'model_file': 'voice_1.pth',
        'index_file': 'voice_1.index',
        'transpose': 0,
    },
    'voice_2': {
        'name': 'Nicușor Dan',
        'model_file': 'voice_2.pth',
        'index_file': 'voice_2.index',
        'transpose': 0,
    },
    'voice_3': {
        'name': 'Diana Șoșoacă',
        'model_file': 'voice_3.pth',
        'index_file': 'voice_3.index',
        'transpose': 0,
    },
}

DEEPFAKE_MODEL_ID = 'MelodyMachine/Deepfake-audio-detection-V2'
# ============================================================

# Verificare ce modele există pe Drive
available_voices = {}
for vid, cfg in VOICE_CONFIG.items():
    model_path = MODELS_DIR / cfg['model_file']
    status = '✓' if model_path.exists() else '✗ (lipsește — rulează 02_rvc_finetune.ipynb)'
    print(f'{status} {cfg["name"]} ({vid})')
    if model_path.exists():
        available_voices[vid] = cfg['name']

if not available_voices:
    print('\n⚠ Niciun model găsit. App-ul pornește dar VC nu va funcționa.')
else:
    print(f'\nVoci disponibile: {list(available_voices.values())}')

In [ ]:
# Load deepfake detection model (only once)
import torch
from transformers import AutoFeatureExtractor, AutoModelForAudioClassification

print(f'Loading deepfake model: {DEEPFAKE_MODEL_ID}...')
_df_extractor = AutoFeatureExtractor.from_pretrained(DEEPFAKE_MODEL_ID)
_df_model = AutoModelForAudioClassification.from_pretrained(DEEPFAKE_MODEL_ID)
_df_model.eval()

# Determine which label index corresponds to "fake"
_fake_idx = next(
    (int(i) for i, lbl in _df_model.config.id2label.items()
     if any(k in lbl.lower() for k in ('fake', 'spoof', 'ai', 'synth', 'generated'))),
    1
)
print(f'Model loaded. Labels: {_df_model.config.id2label}')
print(f'Using label index {_fake_idx} as "fake"')

In [ ]:
# ============================================================
# Core function: detect_deepfake()
# ============================================================
import numpy as np
import librosa
import librosa.display
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from typing import Tuple

SAMPLE_RATE_DF = 16000
WINDOW_S = 2.0
HOP_S = 0.5

def _predict_window(window: np.ndarray) -> float:
    inputs = _df_extractor(window, sampling_rate=SAMPLE_RATE_DF, return_tensors='pt', padding=True)
    with torch.no_grad():
        logits = _df_model(**inputs).logits
    probs = torch.softmax(logits, dim=-1).squeeze().numpy()
    return float(probs[_fake_idx])

def detect_deepfake(audio_path: str) -> Tuple[float, plt.Figure, plt.Figure]:
    audio, _ = librosa.load(audio_path, sr=SAMPLE_RATE_DF, mono=True)
    win = int(WINDOW_S * SAMPLE_RATE_DF)
    hop = int(HOP_S * SAMPLE_RATE_DF)
    
    timestamps, scores = [], []
    for start in range(0, max(1, len(audio) - win + 1), hop):
        w = audio[start:start + win]
        if len(w) < win:
            w = np.pad(w, (0, win - len(w)))
        scores.append(_predict_window(w))
        timestamps.append((start + win // 2) / SAMPLE_RATE_DF)
    
    if not scores:
        padded = np.pad(audio, (0, win - len(audio)))
        scores = [_predict_window(padded)]
        timestamps = [len(audio) / (2 * SAMPLE_RATE_DF)]
    
    ts = np.array(timestamps)
    fs = np.array(scores)
    weights = np.hanning(len(fs)) + 0.1
    global_score = float(np.average(fs, weights=weights))
    
    # Timeline plot
    fig_tl, ax = plt.subplots(figsize=(10, 3))
    ax.fill_between(ts, fs, alpha=0.25, color='#e74c3c')
    ax.plot(ts, fs, color='#e74c3c', linewidth=1.8, label='Frame score')
    ax.axhline(y=global_score, color='#c0392b', linestyle='--', linewidth=1.5,
               label=f'Global: {global_score:.1%}')
    ax.axhline(y=0.5, color='gray', linestyle=':', linewidth=1, alpha=0.7)
    ax.set_ylim(0, 1)
    ax.set_xlabel('Timp (s)')
    ax.set_ylabel('P(AI-generat)')
    verdict = '🔴 AI-generat' if global_score > 0.5 else '🟢 Real'
    ax.set_title(f'Timeline Deepfake — {verdict} ({global_score:.1%})')
    ax.legend()
    fig_tl.tight_layout()
    
    # Spectrogram + overlay plot
    fig_sp, (ax_s, ax_p) = plt.subplots(2, 1, figsize=(12, 6),
                                          gridspec_kw={'height_ratios': [3, 1]}, sharex=True)
    S = librosa.feature.melspectrogram(y=audio, sr=SAMPLE_RATE_DF, n_mels=128, fmax=8000)
    S_dB = librosa.power_to_db(S, ref=np.max)
    img = librosa.display.specshow(S_dB, sr=SAMPLE_RATE_DF, x_axis='time', y_axis='mel',
                                    fmax=8000, ax=ax_s, cmap='magma')
    fig_sp.colorbar(img, ax=ax_s, format='%+2.0f dB')
    for t, score in zip(ts, fs):
        ax_s.axvspan(t - HOP_S / 2, t + HOP_S / 2, alpha=0.30,
                     color=plt.cm.RdYlGn_r(score), linewidth=0)
    ax_s.set_title('Mel Spectrogram cu Overlay (roșu = suspect AI)')
    ax_p.plot(ts, fs, color='#e74c3c', linewidth=1.5)
    ax_p.fill_between(ts, fs, alpha=0.25, color='#e74c3c')
    ax_p.axhline(y=0.5, color='gray', linestyle=':', linewidth=1, alpha=0.7)
    ax_p.set_ylim(0, 1)
    ax_p.set_ylabel('P(fake)')
    ax_p.set_xlabel('Timp (s)')
    fig_sp.tight_layout()
    
    return global_score, fig_tl, fig_sp

print('detect_deepfake() defined.')

In [ ]:
# ============================================================
# Core function: voice_convert()
# ============================================================
import soundfile as sf
import tempfile
import numpy as np
import os

_rvc_instance = None
_rvc_current_voice = None

def voice_convert(
    audio_path: str,
    voice_id: str,
    transpose: int = 0,
    index_rate: float = 0.75,
    protect: float = 0.33,
) -> str:
    global _rvc_instance, _rvc_current_voice

    if voice_id not in VOICE_CONFIG:
        raise ValueError(f'Unknown voice: {voice_id}')

    cfg = VOICE_CONFIG[voice_id]
    model_path = str(MODELS_DIR / cfg['model_file'])
    index_path = str(MODELS_DIR / cfg['index_file'])

    if not Path(model_path).exists():
        raise FileNotFoundError(f'Model negasit: {model_path}')

    if _rvc_current_voice != voice_id:
        os.chdir(RVC_DIR)
        _orig_argv = sys.argv[:]
        sys.argv = ['']          # argparse din Config() citeste sys.argv; in Colab are args Jupyter
        from infer.modules.vc.modules import VC
        from configs.config import Config
        cfg_rvc = Config()
        sys.argv = _orig_argv
        _rvc_instance = VC(cfg_rvc)
        _rvc_instance.get_vc(model_path)
        _rvc_current_voice = voice_id
        print(f'Model incarcat: {cfg["name"]}')

    use_index = index_path if Path(index_path).exists() else ''
    pitch = transpose + cfg.get('transpose', 0)

    result = _rvc_instance.vc_single(
        sid=0,
        input_audio_path=audio_path,
        f0_up_key=pitch,
        f0_file=None,
        f0_method='rmvpe',
        file_index=use_index,
        index_rate=index_rate,
        filter_radius=3,
        resample_sr=0,
        rms_mix_rate=0.25,
        protect=protect,
    )

    if isinstance(result, (tuple, list)) and len(result) == 2:
        tgt_sr, output_audio = result
    else:
        raise RuntimeError(f'Format necunoscut: {type(result)}')

    if output_audio is None or len(output_audio) == 0:
        raise RuntimeError('Audio gol returnat de model.')

    output_audio = np.array(output_audio, dtype=np.float32)
    tmp = tempfile.NamedTemporaryFile(suffix='.wav', delete=False)
    sf.write(tmp.name, output_audio, int(tgt_sr))
    return tmp.name

print('voice_convert() definita.')

In [ ]:
# ============================================================
# Gradio UI — Gradio callback functions
# ============================================================
import gradio as gr

VOICE_CHOICES = [cfg['name'] for cfg in VOICE_CONFIG.values()]
VOICE_ID_BY_NAME = {cfg['name']: vid for vid, cfg in VOICE_CONFIG.items()}


def vc_callback(audio_input, voice_name, transpose, index_rate, protect):
    """Gradio callback for Voice Conversion tab."""
    if audio_input is None:
        return None, '⚠ Niciun fișier audio. Uploadează sau înregistrează un .wav.'
    
    voice_id = VOICE_ID_BY_NAME.get(voice_name)
    if voice_id is None:
        return None, f'⚠ Voce necunoscută: {voice_name}'
    
    model_path = MODELS_DIR / VOICE_CONFIG[voice_id]['model_file']
    if not model_path.exists():
        return None, (
            f'⚠ Modelul pentru "{voice_name}" nu există încă.\n'
            'Rulează 02_rvc_finetune.ipynb pentru a antrena modelele.'
        )
    
    try:
        output_path = voice_convert(
            audio_input, voice_id,
            transpose=int(transpose),
            index_rate=float(index_rate),
            protect=float(protect),
        )
        return output_path, f'✓ Conversie completă → {voice_name}'
    except Exception as e:
        return None, f'✗ Eroare: {str(e)}'


def df_callback(audio_input):
    """Gradio callback for Deepfake Detection tab."""
    if audio_input is None:
        return None, None, '⚠ Niciun fișier audio.'
    
    try:
        score, fig_tl, fig_sp = detect_deepfake(audio_input)
        if score > 0.75:
            verdict = f'🔴 PROBABIL AI-GENERAT ({score:.1%})'
        elif score > 0.5:
            verdict = f'🟡 POSIBIL AI-GENERAT ({score:.1%})'
        elif score > 0.25:
            verdict = f'🟡 POSIBIL REAL ({score:.1%})'
        else:
            verdict = f'🟢 PROBABIL REAL ({score:.1%})'
        
        disclaimer = '\n⚠ Acuratețea pe limba română nu este validată științific.'
        return fig_tl, fig_sp, verdict + disclaimer
    except Exception as e:
        return None, None, f'✗ Eroare: {str(e)}'


print('Gradio callbacks defined.')

In [ ]:
# ============================================================
# Gradio UI — Layout redesenat
# ============================================================
import gradio as gr

CUSTOM_CSS = """
.step-label {
    font-size: 0.85em;
    font-weight: 700;
    color: #6c63ff;
    text-transform: uppercase;
    letter-spacing: 0.08em;
    margin-bottom: 4px;
}
.voice-radio label {
    border: 2px solid #e2e8f0;
    border-radius: 10px;
    padding: 10px 16px;
    margin: 4px 0;
    transition: border-color 0.2s;
}
.voice-radio label:hover {
    border-color: #6c63ff;
}
.verdict-box textarea {
    font-size: 1.3em !important;
    font-weight: 700;
    text-align: center;
    background: #f8f9fa;
    border-radius: 10px;
}
.convert-btn {
    height: 56px !important;
    font-size: 1.1em !important;
    border-radius: 12px !important;
}
footer { display: none !important; }
"""

VOICE_DESCRIPTIONS = {
    'voice_1': ('Calin Georgescu',   'Politician — ton grav, cadenta lenta'),
    'voice_2': ('Nicusor Dan',       'Primar Bucuresti — ton academic, clar'),
    'voice_3': ('Diana Sosoaca',     'Politician — ton ferm, energic'),
}

RADIO_CHOICES = [
    f'{name}  —  {desc}'
    for vid, (name, desc) in VOICE_DESCRIPTIONS.items()
    if vid in {v: None for v in VOICE_CONFIG}
]
RADIO_TO_VOICE_ID = {
    f'{name}  —  {desc}': vid
    for vid, (name, desc) in VOICE_DESCRIPTIONS.items()
}


def vc_callback_radio(audio_input, radio_choice, transpose, index_rate, protect):
    if audio_input is None:
        return None, 'Incarca un fisier audio mai intai.'
    if radio_choice is None:
        return None, 'Selecteaza o voce tinta.'
    voice_id = RADIO_TO_VOICE_ID.get(radio_choice)
    if voice_id is None:
        return None, f'Voce necunoscuta: {radio_choice}'
    model_path = MODELS_DIR / VOICE_CONFIG[voice_id]['model_file']
    if not model_path.exists():
        return None, f'Modelul pentru aceasta voce lipseste.\nRuleaza 02_rvc_finetune.ipynb intai.'
    try:
        out = voice_convert(audio_input, voice_id,
                            transpose=int(transpose),
                            index_rate=float(index_rate),
                            protect=float(protect))
        name = VOICE_DESCRIPTIONS[voice_id][0]
        return out, f'Conversie completa → {name}'
    except Exception as e:
        return None, f'Eroare: {str(e)}'


def df_callback_new(audio_input):
    if audio_input is None:
        return None, None, 'Incarca un fisier audio.'
    try:
        score, fig_tl, fig_sp = detect_deepfake(audio_input)
        if score > 0.75:
            verdict = f'PROBABIL AI-GENERAT\n{score:.1%} probabilitate fake'
        elif score > 0.5:
            verdict = f'POSIBIL AI-GENERAT\n{score:.1%} probabilitate fake'
        elif score > 0.25:
            verdict = f'POSIBIL REAL\n{score:.1%} probabilitate fake'
        else:
            verdict = f'PROBABIL REAL\n{score:.1%} probabilitate fake'
        return fig_tl, fig_sp, verdict
    except Exception as e:
        return None, None, f'Eroare: {str(e)}'


with gr.Blocks(
    theme=gr.themes.Soft(primary_hue='violet', neutral_hue='slate'),
    title='Neural Voice Identity Control',
    css=CUSTOM_CSS,
) as app:

    # ── HEADER ─────────────────────────────────────────────────
    gr.HTML("""
    <div style="text-align:center; padding:28px 0 12px 0;">
      <h1 style="font-size:2em; font-weight:800; margin:0; color:#1e293b;">
        Neural Voice Identity Control
      </h1>
      <p style="color:#64748b; margin:6px 0 0 0; font-size:1em;">
        Deepfake Audio Analysis System &nbsp;·&nbsp; Proiect academic 2026
      </p>
    </div>
    """)

    with gr.Tabs():

        # ── TAB 1: VOICE CONVERSION ─────────────────────────────
        with gr.Tab('🎭  Conversie Vocala'):

            with gr.Row(equal_height=False):

                # LEFT — input steps
                with gr.Column(scale=1, min_width=320):

                    gr.HTML('<p class="step-label">Pasul 1 — Incarca audio</p>')
                    vc_input = gr.Audio(
                        label='',
                        sources=['upload', 'microphone'],
                        type='filepath',
                    )

                    gr.HTML('<p class="step-label" style="margin-top:18px;">Pasul 2 — Alege vocea tinta</p>')
                    vc_voice_radio = gr.Radio(
                        choices=RADIO_CHOICES,
                        value=RADIO_CHOICES[0] if RADIO_CHOICES else None,
                        label='',
                        elem_classes='voice-radio',
                    )

                    with gr.Accordion('Setari avansate', open=False):
                        vc_transpose = gr.Slider(
                            label='Pitch shift (semitonuri)',
                            minimum=-12, maximum=12, step=1, value=0,
                            info='0 = pastreaza inaltimea originala'
                        )
                        vc_index_rate = gr.Slider(
                            label='Index rate',
                            minimum=0.0, maximum=1.0, step=0.05, value=0.75,
                            info='Cat de mult se foloseste FAISS index-ul (0.75 recomandat)'
                        )
                        vc_protect = gr.Slider(
                            label='Consonant protection',
                            minimum=0.0, maximum=0.5, step=0.01, value=0.33,
                            info='Protejeaza consoanele de artefacte (0.33 recomandat)'
                        )

                    vc_btn = gr.Button(
                        '🎭  Converteste vocea',
                        variant='primary',
                        size='lg',
                        elem_classes='convert-btn',
                    )

                # RIGHT — output
                with gr.Column(scale=1, min_width=320):
                    gr.HTML('<p class="step-label">Pasul 3 — Asculta si descarca</p>')
                    vc_output = gr.Audio(
                        label='',
                        type='filepath',
                        interactive=False,
                        show_download_button=True,
                    )
                    vc_status = gr.Textbox(
                        label='Status',
                        interactive=False,
                        lines=2,
                        placeholder='Asteapta conversia...',
                    )

            vc_btn.click(
                fn=vc_callback_radio,
                inputs=[vc_input, vc_voice_radio, vc_transpose, vc_index_rate, vc_protect],
                outputs=[vc_output, vc_status],
            )

        # ── TAB 2: DEEPFAKE DETECTION ───────────────────────────
        with gr.Tab('🔍  Detectie Deepfake'):

            with gr.Row(equal_height=False):

                with gr.Column(scale=1, min_width=300):
                    gr.HTML('<p class="step-label">Pasul 1 — Incarca audio</p>')
                    df_input = gr.Audio(
                        label='',
                        sources=['upload', 'microphone'],
                        type='filepath',
                    )
                    df_btn = gr.Button(
                        '🔍  Analizeaza',
                        variant='primary',
                        size='lg',
                        elem_classes='convert-btn',
                    )
                    gr.HTML('<p class="step-label" style="margin-top:18px;">Verdict</p>')
                    df_verdict = gr.Textbox(
                        label='',
                        interactive=False,
                        lines=3,
                        elem_classes='verdict-box',
                        placeholder='Asteapta analiza...',
                    )

                with gr.Column(scale=2):
                    df_timeline = gr.Plot(label='Analiza temporala — probabilitate per cadru')
                    df_spectrogram = gr.Plot(label='Spectrograma Mel cu overlay deepfake')

            df_btn.click(
                fn=df_callback_new,
                inputs=[df_input],
                outputs=[df_timeline, df_spectrogram, df_verdict],
            )

        # ── TAB 3: DESPRE ───────────────────────────────────────
        with gr.Tab('ℹ️  Despre'):
            gr.Markdown("""
## Despre sistem

Proiect practic pentru lucrarea de disertatie:
**Neural Voice Identity Control and Deepfake Audio Analysis System** (2026).

### Tab Voice Conversion
Transforma timbrul vocal al unui fisier audio in vocea uneia din cele 3 personalitati
romanesti pre-antrenate. Tehnologie: **RVC v2** (Retrieval-based Voice Conversion).
Functioneaza pe orice limba vorbita, nu doar romana.

### Tab Deepfake Detection
Analizeaza daca un fisier audio este generat de AI sau este real.
Produce scor global + grafic temporal + spectrograma cu overlay.
Tehnologie: model **wav2vec2** pre-antrenat (`MelodyMachine/Deepfake-audio-detection-V2`).

### Limitari cunoscute
- Calitatea vocii convertite depinde de cantitatea si calitatea datelor de antrenare.
- Detectorul este antrenat predominant pe audio englezesc — pe romana poate fi mai putin precis.
- Inferenta poate dura 10–30 de secunde pe T4 GPU.

### Disclaimer
Vocile simulate sunt ale unor persoane publice (inregistrari disponibile public).
Sistemul este construit **exclusiv in scop academic**.
Utilizarea pentru inselaciune sau uz comercial este interzisa.
            """)

print('Gradio app definita. Ruleaza celula urmatoare pentru lansare.')


In [ ]:
# LANSARE APP
app.queue().launch(
    share=True,
    debug=False,
    quiet=False,
)